In [0]:
# ============================================================
# NOTEBOOK : DIM_INVENTORY
# PURPOSE  : INVENTORY DIMENSION INCREMENTAL LOAD
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid

In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("inventory_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.dim_inventory"

    pipeline_name = "PL_DIM_INVENTORY"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            inventory_id,
            product_id,
            warehouse_location,
            available_stock,
            reorder_level,
            stock_last_updated,
            created_date,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_inventory"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : inventory_tbl
Rows Read : 5


Max Date updated successfully


FN_LOGGER LOADED SUCCESSFULLY


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            i.inventory_id,

            i.product_id,

            p.product_name,

            i.warehouse_location,

            i.available_stock,

            i.reorder_level,

            i.stock_last_updated,

            CASE

                WHEN i.available_stock <= i.reorder_level
                THEN 'REORDER_REQUIRED'

                WHEN i.available_stock <= 50
                THEN 'LOW_STOCK'

                ELSE 'AVAILABLE'

            END AS stock_status,

            CASE

                WHEN i.warehouse_location = 'WH_CHN'
                THEN 'CHENNAI'

                WHEN i.warehouse_location = 'WH_BLR'
                THEN 'BANGALORE'

                WHEN i.warehouse_location = 'WH_MUM'
                THEN 'MUMBAI'

                ELSE 'OTHER'

            END AS warehouse_region,

            i.modified_date,

            sha2(
                concat_ws(
                    '|',
                    i.product_id,
                    p.product_name,
                    i.warehouse_location,
                    i.available_stock,
                    i.reorder_level
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_inventory i

        LEFT JOIN silver.dim_product p
            ON i.product_id = p.product_id
            AND p.is_current = 1

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_inventory"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)



Silver View Created : inventory_tbl


In [0]:
# ============================================================
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE IF NOT EXISTS silver.dim_inventory
        (
            inventory_id BIGINT,
            product_id BIGINT,
            product_name STRING,
            warehouse_location STRING,
            available_stock BIGINT,
            reorder_level BIGINT,
            stock_last_updated TIMESTAMP,
            stock_status STRING,
            warehouse_region STRING,
            modified_date TIMESTAMP,
            hash_key STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current INT,
            is_deleted INT
        )

        USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.dim_inventory


In [0]:
# ============================================================
# MERGE LOGIC
# ============================================================

try:

    spark.sql("""

        MERGE INTO silver.dim_inventory AS target

        USING vw_silver_inventory AS source

        ON target.inventory_id = source.inventory_id
           AND target.is_current = 1

        WHEN MATCHED
             AND target.hash_key <> source.hash_key

        THEN UPDATE SET

            target.effective_end_date =
                current_timestamp(),

            target.is_current = 0

        WHEN NOT MATCHED

        THEN INSERT
        (
            inventory_id,
            product_id,
            product_name,
            warehouse_location,
            available_stock,
            reorder_level,
            stock_last_updated,
            stock_status,
            warehouse_region,
            modified_date,
            hash_key,
            effective_start_date,
            effective_end_date,
            is_current,
            is_deleted
        )

        VALUES
        (
            source.inventory_id,
            source.product_id,
            source.product_name,
            source.warehouse_location,
            source.available_stock,
            source.reorder_level,
            source.stock_last_updated,
            source.stock_status,
            source.warehouse_region,
            source.modified_date,
            source.hash_key,
            source.effective_start_date,
            source.effective_end_date,
            source.is_current,
            source.is_deleted
        )

    """)

    print(f"Merge Completed : {table_name}")

    spark.sql("""

        UPDATE silver.dim_inventory

        SET

            is_deleted = 1,
            is_current = 0,
            effective_end_date = current_timestamp()

        WHERE inventory_id NOT IN
        (
            SELECT inventory_id
            FROM vw_silver_inventory
        )

        AND is_current = 1

    """)

    print(f"Soft Delete Completed : {table_name}")

except Exception as e:

    print(f"Merge Failed : {table_name}")

    raise(e)



Merge Completed : inventory_tbl
Soft Delete Completed : inventory_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"DIM_INVENTORY SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "DIM_INVENTORY",
        str(e)

    )

    print(f"DIM_INVENTORY LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : inventory_tbl
Watermark Updated : inventory_tbl
Audit Log Inserted : inventory_tbl
DIM_INVENTORY SUCCESSFULLY LOADED : inventory_tbl
